# Feature ML Lab - LGBM Regression

Select features for the LightGBM regression stack using Spearman as the primary optimisation target.
This follows the same workflow as the feature classification lab, but it is tuned for the regression
targets used by `2_1 Train LGBM only.ipynb`.

In [1]:
import os
import json
import time
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from IPython.display import display

warnings.filterwarnings("ignore")

from Learn.features import add_feature_library
from Learn.labels import calculate_trade_outcomes_all_candles
from Learn.preprocess import preprocess_ohlcv

In [2]:
def load_ohlcv(ds_name, start_date=None, n_rows=None):
    """Load an OHLCV CSV, optionally sliced by start_date or tail n_rows."""
    df = pd.read_csv(ds_name)
    df = df.sort_values("Time").reset_index(drop=True)
    df["Time"] = pd.to_datetime(df["Time"])
    if start_date is not None:
        df = df[df["Time"] >= pd.Timestamp(start_date)].reset_index(drop=True)
    if n_rows is not None:
        df = df.tail(n_rows).reset_index(drop=True)
    return df


def _spearman_metric(y_true, y_pred):
    value = stats.spearmanr(y_true, y_pred, nan_policy="omit").statistic
    return 0.0 if value is None or np.isnan(value) else float(value)


def _session_seq_indices(times, seq_len, trading_hours):
    if trading_hours is None:
        return list(range(len(times) - seq_len))
    t_start = pd.to_datetime(trading_hours[0]).time()
    t_end = pd.to_datetime(trading_hours[1]).time()
    return [
        i for i in range(len(times) - seq_len)
        if t_start <= pd.Timestamp(times[i + seq_len - 1]).time() < t_end
    ]

In [3]:
# Dataset and feature-selection configuration
DS_NAME = "../data/US500_M1_520weeks.csv"
SYMBOL = "US500"
N_ROWS = 1_000_000

INCLUDE_MTF = True
FAST_MODE = False
SPLIT_RATIO = 0.80
VAR_THRESHOLD = 0.001
CORR_THRESHOLD = 0.85

# Regression targets from the LGBM training plan
TP_MULT = 2.5
SL_MULT = 2.5
TARGET_CLIP_MAX = 2.0
TARGET_COLS = ["long_quality", "short_quality", "signed_quality"]
ALL_TARGET_COLS = TARGET_COLS + ["tradeability_score"]

# Labelling / regime parameters
regime_params = {
    "ma_period": 90,
    "slope_smoothness": 50,
    "regime_min_duration": 0,
    "atr_window": 60,
    "atr_lookback": 720,
    "atr_percentile": 0.0,
    "slope_threshold": 0,
}

outcome_params = {
    "atr_window": 60,
    "tp_mult": TP_MULT,
    "sl_mult": SL_MULT,
    "max_horizon": 30,
}

# LGBM feature-selection / evaluation settings
LGBM_PARAMS = {
    "objective": "regression",
    "n_estimators": 600,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.5,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}

N_SELECT_PER_MODEL = 30
N_PERM_REPEATS = 5

device = "cpu"
print("Dataset:", DS_NAME)
print("Targets:", TARGET_COLS)

Dataset: ../data/US500_M1_520weeks.csv
Targets: ['long_quality', 'short_quality', 'signed_quality']


In [4]:
df = load_ohlcv(DS_NAME, n_rows=N_ROWS)
print(f"Loaded {len(df):,} rows")
print(df["Time"].min(), "→", df["Time"].max())

df = add_feature_library(df.copy(), include_mtf=INCLUDE_MTF, regime_params=regime_params)

from talib import ATR

df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)

outcomes = calculate_trade_outcomes_all_candles(df, **outcome_params)

eps = 1e-8
long_q = np.log1p(outcomes["buy_MFE"] / (outcomes["buy_MAE"] + eps))
short_q = np.log1p(outcomes["sell_MFE"] / (outcomes["sell_MAE"] + eps))

if TARGET_CLIP_MAX is not None:
    long_q = np.clip(long_q, 0.0, TARGET_CLIP_MAX)
    short_q = np.clip(short_q, 0.0, TARGET_CLIP_MAX)

df["long_quality"] = long_q
df["short_quality"] = short_q
df["signed_quality"] = df["long_quality"] - df["short_quality"]
df["tradeability_score"] = 1.0

print(
    f"Target stats | long std={df['long_quality'].std():.4f} "
    f"short std={df['short_quality'].std():.4f} "
    f"signed std={df['signed_quality'].std():.4f}"
)
print(
    f"Target sparsity | long zero={(df['long_quality'] <= 1e-12).mean():.3f} "
    f"short zero={(df['short_quality'] <= 1e-12).mean():.3f}"
)

Loaded 1,000,000 rows
2023-07-31 15:37:00+00:00 → 2026-06-19 16:59:00+00:00
Target stats | long std=0.6314 short std=0.6233 signed std=1.1718
Target sparsity | long zero=0.235 short zero=0.244


In [5]:
preprocess_ohlcv_args = {
    "target_col": ALL_TARGET_COLS,
    "outcomes_col": None,
    "shift": 0,
    "onehot_prefixes": ["OH_"],
    "price_prefixes": ["PR_"],
}

split_idx = int(len(df) * SPLIT_RATIO)
df_train = df.iloc[:split_idx].copy()
df_test = df.iloc[split_idx:].copy()

X_train, y_train, scaler, feature_cols_full, _, proc_df_train = preprocess_ohlcv(
    df_train,
    **preprocess_ohlcv_args,
    scaler=None,
    return_df=True,
)
X_test, y_test, _, _, _, proc_df_test = preprocess_ohlcv(
    df_test,
    **preprocess_ohlcv_args,
    scaler=scaler,
    return_df=True,
)

target_arrays = {
    name: {
        "train": y_train[:, i],
        "test": y_test[:, i],
    }
    for i, name in enumerate(TARGET_COLS)
}

print(f"Train rows={len(X_train):,} | Test rows={len(X_test):,} | Features={len(feature_cols_full)}")
print("Train target shape:", y_train.shape)
print("Test target shape :", y_test.shape)

Train rows=799,252 | Test rows=199,999 | Features=199
Train target shape: (799252, 4)
Test target shape : (199999, 4)


In [6]:
from sklearn.feature_selection import mutual_info_regression

# Step 1: variance filter fitted on train only
_var_train = X_train.var(axis=0)
_var_mask = _var_train >= VAR_THRESHOLD
_cols_after_var = [c for c, ok in zip(feature_cols_full, _var_mask) if ok]
_n_dropped_var = len(feature_cols_full) - len(_cols_after_var)
X_train_v = X_train[:, _var_mask]
X_test_v = X_test[:, _var_mask]
print(f"Variance filter: {_n_dropped_var} features dropped -> {len(_cols_after_var)} remaining")

# Step 2: MI ranking per regression target
print("Computing mutual information on training data ...")
_t0 = time.time()
mi_scores = {}
for target_name in TARGET_COLS:
    mi_scores[target_name] = mutual_info_regression(
        X_train_v,
        target_arrays[target_name]["train"],
        random_state=42,
    )
print(f"  Done in {time.time() - _t0:.1f}s")

mi_df = pd.DataFrame({"feature": _cols_after_var})
for target_name in TARGET_COLS:
    mi_df[f"mi_{target_name}"] = mi_scores[target_name]
mi_df["mi_mean"] = mi_df[[f"mi_{t}" for t in TARGET_COLS]].mean(axis=1)
mi_df["mi_max"] = mi_df[[f"mi_{t}" for t in TARGET_COLS]].max(axis=1)
mi_df = mi_df.sort_values("mi_mean", ascending=False).reset_index(drop=True)
mi_df.index += 1

# Step 3: correlation dedup using mean MI order
print(f"Correlation dedup (threshold |r| > {CORR_THRESHOLD}) ...")
_corr_mat = np.corrcoef(X_train_v.T)
_feat_order = mi_df["feature"].tolist()
_feat_idx = {f: i for i, f in enumerate(_cols_after_var)}

dropped_by_corr = set()
corr_pairs_dropped = []

for _i, _fi in enumerate(_feat_order):
    if _fi in dropped_by_corr:
        continue
    _ci = _feat_idx[_fi]
    for _fj in _feat_order[_i + 1:]:
        if _fj in dropped_by_corr:
            continue
        _cj = _feat_idx[_fj]
        if abs(_corr_mat[_ci, _cj]) > CORR_THRESHOLD:
            dropped_by_corr.add(_fj)
            corr_pairs_dropped.append((_fi, _fj, _corr_mat[_ci, _cj]))

prescreened_features = [f for f in _feat_order if f not in dropped_by_corr]
print(f"  Corr dedup: {len(dropped_by_corr)} features dropped -> {len(prescreened_features)} remaining")

if corr_pairs_dropped:
    print("Dropped pairs (first 10):")
    for _a, _b, _r in sorted(corr_pairs_dropped, key=lambda x: -abs(x[2]))[:10]:
        print(f"  keep: {_a:<40} drop: {_b:<40} r={_r:+.3f}")

_prescreened_idx = [_cols_after_var.index(f) for f in prescreened_features]
X_train_ps = X_train_v[:, _prescreened_idx]
X_test_ps = X_test_v[:, _prescreened_idx]
mi_ps_df = mi_df[mi_df["feature"].isin(prescreened_features)].reset_index(drop=True)
mi_ps_df.index += 1

print(f"Raw features      : {len(feature_cols_full)}")
print(f"After variance    : {len(_cols_after_var)}")
print(f"After corr dedup  : {len(prescreened_features)}")
print()
print(mi_ps_df[["feature", "mi_mean", "mi_max"]].head(20).to_string())

Variance filter: 6 features dropped -> 193 remaining
Computing mutual information on training data ...
  Done in 5993.4s
Correlation dedup (threshold |r| > 0.85) ...
  Corr dedup: 74 features dropped -> 118 remaining
Dropped pairs (first 10):
  keep: fl_roc_5                                 drop: fl_ret_5                                 r=+1.000
  keep: fl_ret_10                                drop: fl_roc_10                                r=+1.000
  keep: fl_ret_20                                drop: fl_roc_20                                r=+1.000
  keep: fl_squeeze                               drop: fl_squeeze                               r=+1.000
  keep: fl_rsi_14                                drop: fl_cmo_14                                r=+1.000
  keep: fl_stoch_k_lag_1                         drop: fl_stoch_d                               r=+0.995
  keep: fl_O_rel                                 drop: fl_body                                  r=-0.994
  keep: fl_parkinson_2

In [7]:
# MI ranking and feature correlation diagnostics
_N_SHOW = min(40, len(mi_ps_df))
_top_mi = mi_ps_df.head(_N_SHOW)
_mi_max = float(_top_mi["mi_mean"].max()) if len(_top_mi) else 0.0
_bar_colors = [
    f"rgba(130,170,255,{0.35 + 0.65 * v / (_mi_max + 1e-9):.2f})"
    for v in _top_mi["mi_mean"]
]

fig = go.Figure(go.Bar(
    y=_top_mi["feature"][::-1],
    x=_top_mi["mi_mean"][::-1],
    orientation="h",
    marker_color=_bar_colors[::-1],
    text=[f"{v:.4f}" for v in _top_mi["mi_mean"][::-1]],
    textposition="outside",
))
fig.update_layout(
    title=f"MI Ranking - Top {_N_SHOW} Pre-screened Features (mean over regression targets)",
    height=max(500, _N_SHOW * 22),
    xaxis=dict(title="MI Score", gridcolor="#333"),
    yaxis=dict(automargin=True, gridcolor="#333"),
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(color="#d4d4d4"),
    margin=dict(l=200, r=80, t=60, b=40),
)
fig.show()

_N_CORR = min(30, len(mi_ps_df))
_hmap_fs = mi_ps_df["feature"].head(_N_CORR).tolist()
_hmap_ix = [_prescreened_idx[prescreened_features.index(f)] for f in _hmap_fs]
_X_hmap = X_train_v[:, _hmap_ix]
_corr_h = np.corrcoef(_X_hmap.T)

fig2 = go.Figure(go.Heatmap(
    z=_corr_h,
    x=_hmap_fs,
    y=_hmap_fs,
    colorscale="RdBu_r",
    zmid=0,
    zmin=-1,
    zmax=1,
    colorbar=dict(title="r"),
    text=[[f"{v:.2f}" for v in row] for row in _corr_h],
    texttemplate="%{text}",
    textfont=dict(size=7),
))
fig2.update_layout(
    title=f"Correlation Matrix - Top {_N_CORR} Pre-screened Features",
    height=700,
    width=730,
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(color="#d4d4d4"),
    xaxis=dict(tickangle=45, automargin=True),
    yaxis=dict(automargin=True),
    margin=dict(l=150, r=20, t=60, b=150),
)
fig2.show()

In [8]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.inspection import permutation_importance

def _reg_metrics(y_true, y_pred):
    spearman = _spearman_metric(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
        "Spearman": spearman,
    }

spearman_scorer = make_scorer(_spearman_metric, greater_is_better=True)

bench_results_full = {}
fitted_models_full = {}
feat_rankings = {}

for target_name in TARGET_COLS:
    print(f"Training full-feature LGBM for {target_name} ...", end="  ")
    model = LGBMRegressor(**LGBM_PARAMS)
    _t0 = time.time()
    model.fit(X_train_ps, target_arrays[target_name]["train"])
    print(f"{time.time() - _t0:.1f}s")

    pred = model.predict(X_test_ps)
    bench_results_full[target_name] = _reg_metrics(target_arrays[target_name]["test"], pred)
    fitted_models_full[target_name] = model
    feat_rankings[f"{target_name}_lgbm"] = pd.Series(model.feature_importances_, index=prescreened_features)

    print(f"Permutation importance for {target_name} ...", end="  ")
    _t0 = time.time()
    pi = permutation_importance(
        model,
        X_test_ps,
        target_arrays[target_name]["test"],
        n_repeats=N_PERM_REPEATS,
        random_state=42,
        n_jobs=-1,
        scoring=spearman_scorer,
    )
    feat_rankings[f"{target_name}_perm"] = pd.Series(
        np.maximum(pi.importances_mean, 0.0),
        index=prescreened_features,
    )
    print(f"{time.time() - _t0:.1f}s")

bench_df_full = pd.DataFrame(bench_results_full).T.round(4)
display(bench_df_full)

Training full-feature LGBM for long_quality ...  42.8s
Permutation importance for long_quality ...  930.5s
Training full-feature LGBM for short_quality ...  42.7s
Permutation importance for short_quality ...  940.6s
Training full-feature LGBM for signed_quality ...  44.3s
Permutation importance for signed_quality ...  897.6s


,MAE,RMSE,R2,Spearman
long_quality,0.4956,0.5874,0.0886,0.3021
short_quality,0.4916,0.5843,0.0854,0.2967
signed_quality,0.9468,1.1046,0.0703,0.2543


In [9]:
# Merge rankings from model importances and Spearman-based permutation scores
_selection_votes = {}
for _src, _imp in feat_rankings.items():
    _top_n = _imp.nlargest(N_SELECT_PER_MODEL).index.tolist()
    for _f in _top_n:
        _selection_votes[_f] = _selection_votes.get(_f, 0) + 1

merged_rank = pd.Series(_selection_votes).sort_values(ascending=False).rename("vote_count")
_avg_imp = pd.concat(
    [_s / (_s.max() + 1e-9) for _s in feat_rankings.values()],
    axis=1,
).mean(axis=1)

merged_rank_df = (
    pd.DataFrame({"vote_count": merged_rank, "avg_norm_imp": _avg_imp})
    .dropna()
    .sort_values(["vote_count", "avg_norm_imp"], ascending=False)
)

selected_features_all = merged_rank_df.index.tolist()
print(f"Initial selected pool: {len(selected_features_all)} features")
display(merged_rank_df.head(30))

# Sweep feature counts to maximise mean Spearman on the validation split
max_sweep = min(len(selected_features_all), 150)
candidate_sizes = sorted({
    int(v)
    for v in np.linspace(max(10, min(10, max_sweep)), max_sweep, num=min(8, max_sweep))
})
if max_sweep not in candidate_sizes:
    candidate_sizes.append(max_sweep)

sweep_rows = []
for n_features in candidate_sizes:
    feat_subset = selected_features_all[:n_features]
    sel_idx = [prescreened_features.index(f) for f in feat_subset]
    X_train_sel_tmp = X_train_ps[:, sel_idx]
    X_test_sel_tmp = X_test_ps[:, sel_idx]

    row = {"n_features": n_features}
    spears = []
    for target_name in TARGET_COLS:
        model = LGBMRegressor(**LGBM_PARAMS)
        model.fit(X_train_sel_tmp, target_arrays[target_name]["train"])
        pred = model.predict(X_test_sel_tmp)
        metrics = _reg_metrics(target_arrays[target_name]["test"], pred)
        row[f"spearman_{target_name}"] = metrics["Spearman"]
        row[f"r2_{target_name}"] = metrics["R2"]
        spears.append(metrics["Spearman"])
    row["mean_spearman"] = float(np.mean(spears))
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).sort_values("mean_spearman", ascending=False).reset_index(drop=True)
best_n_features = int(sweep_df.iloc[0]["n_features"])
selected_features = selected_features_all[:best_n_features]
print(f"Best feature count by mean Spearman: {best_n_features}")
display(sweep_df.round(4))

Initial selected pool: 53 features


,vote_count,avg_norm_imp
MTF_30min_roc,6.0,0.420328
MTF_30min_rsi,6.0,0.403264
MTF_5min_adx,6.0,0.347251
fl_hour_sin,6.0,0.334626
MTF_15min_slope,6.0,0.325673
fl_atr_ratio_14,6.0,0.289503
fl_rv_60,6.0,0.287659
fl_adx14,6.0,0.127095
MTF_30min_adx,5.0,0.500615
fl_close_loc,5.0,0.459725


Best feature count by mean Spearman: 16


,n_features,spearman_long_quality,r2_long_quality,spearman_short_quality,r2_short_quality,spearman_signed_quality,r2_signed_quality,mean_spearman
0,16,0.3041,0.0894,0.2971,0.0858,0.2569,0.0718,0.2860
1,46,0.3029,0.0893,0.2961,0.0851,0.2559,0.0714,0.2850
2,22,0.3023,0.0887,0.2958,0.0850,0.2566,0.0718,0.2849
3,53,0.3022,0.0887,0.2958,0.0849,0.2539,0.0704,0.2840
4,28,0.3030,0.0896,0.2967,0.0853,0.2513,0.0688,0.2837
5,40,0.3019,0.0882,0.2950,0.0842,0.2533,0.0699,0.2834
6,34,0.3009,0.0876,0.2949,0.0842,0.2519,0.0689,0.2826
7,10,0.2680,0.0668,0.2775,0.0711,0.2434,0.0635,0.2630


In [10]:
_N_VIS = min(40, len(selected_features))
_vis_feats = selected_features[:_N_VIS]
_src_names = list(feat_rankings.keys())
_src_colors = [
    "#82aaff", "#c3e88d", "#f78c6c", "#ffcb6b", "#89ddff",
    "#c792ea", "#80cbc4", "#ffab76", "#a5d6a7", "#ef9a9a",
]

fig = go.Figure()
for _si, _src in enumerate(_src_names):
    _imp = feat_rankings[_src]
    _imp_norm = _imp / (_imp.max() + 1e-9)
    _vals = [float(_imp_norm.get(f, 0.0)) for f in _vis_feats]
    fig.add_trace(go.Bar(
        name=_src,
        y=_vis_feats[::-1],
        x=_vals[::-1],
        orientation="h",
        marker_color=_src_colors[_si % len(_src_colors)],
        opacity=0.7,
    ))

fig.update_layout(
    title=f"Feature Ranking - Normalised Importance Across All Model Sources (top {_N_VIS})",
    barmode="stack",
    height=max(600, _N_VIS * 22),
    xaxis=dict(title="Cumulative Normalised Importance", gridcolor="#333"),
    yaxis=dict(automargin=True, gridcolor="#333"),
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(color="#d4d4d4"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=220, r=30, t=80, b=40),
)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sweep_df["n_features"],
    y=sweep_df["mean_spearman"],
    mode="lines+markers",
    name="Mean Spearman",
    line=dict(color="#82aaff", width=3),
))
fig2.update_layout(
    title="Feature Count Sweep - Mean Spearman on Holdout Split",
    xaxis=dict(title="Selected features", gridcolor="#333"),
    yaxis=dict(title="Mean Spearman", gridcolor="#333"),
    height=420,
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(color="#d4d4d4"),
    margin=dict(l=60, r=20, t=60, b=40),
)
fig2.show()

In [11]:
_sel_idx = [prescreened_features.index(f) for f in selected_features]
X_train_sel = X_train_ps[:, _sel_idx]
X_test_sel = X_test_ps[:, _sel_idx]
print(f"Selected feature matrix: {X_train_sel.shape[1]} features")

bench_results_sel = {}
fitted_models_sel = {}
for target_name in TARGET_COLS:
    model = LGBMRegressor(**LGBM_PARAMS)
    print(f"Retraining selected-feature LGBM for {target_name} ...", end="  ")
    _t0 = time.time()
    model.fit(X_train_sel, target_arrays[target_name]["train"])
    print(f"{time.time() - _t0:.1f}s")

    pred = model.predict(X_test_sel)
    bench_results_sel[target_name] = _reg_metrics(target_arrays[target_name]["test"], pred)
    fitted_models_sel[target_name] = model

bench_df_sel = pd.DataFrame(bench_results_sel).T.round(4)

compare_rows = []
for target_name in TARGET_COLS:
    for met in ["MAE", "RMSE", "R2", "Spearman"]:
        compare_rows.append(
            {
                "target": target_name,
                "metric": met,
                "full": round(float(bench_results_full[target_name][met]), 4),
                "selected": round(float(bench_results_sel[target_name][met]), 4),
                "delta": round(
                    float(bench_results_sel[target_name][met] - bench_results_full[target_name][met]),
                    4,
                ),
            }
        )

compare_df = pd.DataFrame(compare_rows)
print(compare_df.pivot_table(index="target", columns="metric", values=["full", "selected", "delta"]).round(4).to_string())

fig = go.Figure()
fig.add_trace(go.Bar(
    name=f"Full ({len(prescreened_features)} features)",
    x=TARGET_COLS,
    y=[bench_results_full[t]["Spearman"] for t in TARGET_COLS],
    marker_color="rgba(130,170,255,0.7)",
))
fig.add_trace(go.Bar(
    name=f"Selected ({len(selected_features)} features)",
    x=TARGET_COLS,
    y=[bench_results_sel[t]["Spearman"] for t in TARGET_COLS],
    marker_color="rgba(195,232,141,0.9)",
))
fig.update_layout(
    title="Spearman - Full vs Selected Features",
    barmode="group",
    yaxis=dict(range=[-1, 1], title="Spearman", gridcolor="#333"),
    xaxis=dict(gridcolor="#333"),
    height=420,
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(color="#d4d4d4"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=60, r=20, t=80, b=40),
)
fig.show()

Selected feature matrix: 16 features
Retraining selected-feature LGBM for long_quality ...  9.5s
Retraining selected-feature LGBM for short_quality ...  10.4s
Retraining selected-feature LGBM for signed_quality ...  8.9s
                 delta                             full                          selected                         
metric             MAE      R2    RMSE Spearman     MAE      R2    RMSE Spearman      MAE      R2    RMSE Spearman
target                                                                                                            
long_quality    0.0003  0.0007 -0.0002   0.0019  0.4956  0.0886  0.5874   0.3021   0.4959  0.0894  0.5871   0.3041
short_quality   0.0000  0.0004 -0.0001   0.0004  0.4916  0.0854  0.5843   0.2967   0.4916  0.0858  0.5842   0.2971
signed_quality -0.0008  0.0014 -0.0009   0.0026  0.9468  0.0703  1.1046   0.2543   0.9459  0.0718  1.1038   0.2569


In [12]:
# Preview selected features and export them as a custom feature function
import textwrap
import re

_FEATURES_PY = os.path.join("Learn", "features.py")
_FUNC_NAME = f"_add_features_{SYMBOL}"

print(f"Selected features ({len(selected_features)}):")
for _i, _f in enumerate(selected_features, 1):
    print(f"  {_i:3d}. {_f}")

with open(_FEATURES_PY, "r", encoding="utf-8") as _fp:
    _existing = _fp.read()

_safe_to_write = f"def {_FUNC_NAME}(" not in _existing
if not _safe_to_write:
    print(f"Warning: function '{_FUNC_NAME}' already exists in {_FEATURES_PY}")
    print("Set OVERWRITE_EXPORT = True below to replace it, or rename SYMBOL.")
else:
    print(f"Function '{_FUNC_NAME}' not found - ready to export.")

OVERWRITE_EXPORT = True

_feat_list_repr = ",\n        ".join([f"'{f}'" for f in selected_features])
_func_source = f'''

def {_FUNC_NAME}(df: pd.DataFrame, include_mtf: bool = False, regime_params: dict = None) -> pd.DataFrame:
    """
    Add the regression features selected for {SYMBOL} by the Feature ML Lab.
    Generated automatically - edit with care.

    Pass regime_params to enable the causal market regime feature (fl_regime).
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
        df = add_feature_library(df, include_mtf=include_mtf, regime_params=regime_params)
    df = df.copy()
    _keep = [
        c for c in [
            {_feat_list_repr},
        ]
        if c in df.columns
    ]
    _ohlcv = [c for c in ["Time", "Open", "High", "Low", "Close", "Volume", "target", "sell_y", "buy_y"]
              if c in df.columns]
    _final = _ohlcv + [c for c in _keep if c not in _ohlcv]
    return df[_final]
'''

print("Function preview (first 20 lines):")
for _i, _line in enumerate(_func_source.splitlines()[:20], 1):
    print(f"  {_i:3d} | {_line}")
print("  ...")

def do_export():
    global _existing
    if not _safe_to_write and not OVERWRITE_EXPORT:
        print(f"Skipped - '{_FUNC_NAME}' already exists. Set OVERWRITE_EXPORT=True to replace.")
        return

    if not _safe_to_write and OVERWRITE_EXPORT:
        _pattern = rf"\n\ndef {re.escape(_FUNC_NAME)}\(.*?(?=\n\ndef |\Z)"
        _existing = re.sub(_pattern, "", _existing, flags=re.DOTALL)

    _new_content = _existing.rstrip() + "\n" + _func_source
    with open(_FEATURES_PY, "w", encoding="utf-8") as _fp:
        _fp.write(_new_content)
    print(f"Written '{_FUNC_NAME}' to {_FEATURES_PY}")
    print(f"Import with: from Learn.features import {_FUNC_NAME}")

do_export()

Selected features (16):
    1. MTF_30min_roc
    2. MTF_30min_rsi
    3. MTF_5min_adx
    4. fl_hour_sin
    5. MTF_15min_slope
    6. fl_atr_ratio_14
    7. fl_rv_60
    8. fl_adx14
    9. MTF_30min_adx
   10. fl_close_loc
   11. MTF_15min_time_in_trend
   12. MTF_30min_time_in_trend
   13. fl_hour_cos
   14. fl_hl_range_atr
   15. fl_ichi_senA_vs_senB
   16. fl_parkinson_20
Set OVERWRITE_EXPORT = True below to replace it, or rename SYMBOL.
Function preview (first 20 lines):
    1 | 
    2 | 
    3 | def _add_features_US500(df: pd.DataFrame, include_mtf: bool = False, regime_params: dict = None) -> pd.DataFrame:
    4 |     """
    5 |     Add the regression features selected for US500 by the Feature ML Lab.
    6 |     Generated automatically - edit with care.
    7 | 
    8 |     Pass regime_params to enable the causal market regime feature (fl_regime).
    9 |     """
   10 |     with warnings.catch_warnings():
   11 |         warnings.simplefilter("ignore", pd.errors.PerformanceWa